# 04 — Feature Extraction for Time Series Forecasting

**Mục tiêu của phần này:**
1. **Trích xuất đa đặc trưng (Multi-feature Extraction):** Thay vì chỉ dùng RMS, ta sẽ trích xuất thêm Variance (Phương sai), Peak-to-Peak (Biên độ đỉnh-đỉnh), và Line Length (Độ dài đường tín hiệu) từ mỗi cửa sổ tín hiệu (window). Việc tính toán được thực hiện bằng cách lấy trung bình trên tất cả các kênh ECoG.
2. **Tạo biến trễ (Lag Features):** Để phục vụ bài toán Dự báo (Forecasting), mô hình cần biết dữ liệu trong quá khứ để dự đoán tương lai. Ta sẽ tạo các biến trễ bậc 1 ($t-1$) và bậc 2 ($t-2$) cho toàn bộ các đặc trưng.

**Đầu ra:** - File `ds003029_forecasting_features.csv` chứa các đặc trưng ở thời điểm $t$ và các biến trễ ở $t-1, t-2$.

### 1. Phân đoạn tín hiệu (Sliding Window)
Trước khi trích xuất đặc trưng, tín hiệu liên tục cần được cắt thành các đoạn nhỏ (cửa sổ). 
- **Kích thước cửa sổ (Window Size):** Khoảng thời gian tín hiệu mà mô hình sẽ "nhìn" vào tại một thời điểm (VD: 2 giây).
- **Bước trượt (Step Size):** Khoảng thời gian cửa sổ dịch chuyển về phía trước (VD: 1 giây). Việc bước trượt nhỏ hơn kích thước cửa sổ sẽ tạo ra sự chồng lấn (overlap), giúp không bỏ sót các biến đổi đột ngột của tín hiệu não.

Kết quả của bước này là danh sách các mảng dữ liệu `windows` và danh sách thời gian trung tâm `t_mids` tương ứng của từng cửa sổ.

In [4]:
import sys
import mne
import numpy as np
import pandas as pd
from pathlib import Path

# =========================================================================
# 1. LOAD ĐƯỜNG DẪN BẰNG MODULE NỘI BỘ (GIỐNG FILE GỐC CỦA BẠN)
# =========================================================================
ws = Path.cwd().resolve()
src_dir = ws / 'src'
if not src_dir.exists() and (ws.parent / 'src').exists():
    ws = ws.parent
src_dir = (ws / 'src').resolve()
sys.path.insert(0, str(src_dir))

try:
    from ds003029_eda.paths import get_paths
except ImportError:
    print("❌ Không tìm thấy module ds003029_eda.paths. Hãy đảm bảo bạn đang chạy notebook ở đúng thư mục project.")

paths = get_paths()

# Load bảng tóm tắt siêu dữ liệu từ Notebook 01
run_summary = pd.read_csv(paths.outputs_dir / 'ds003029_run_summary.csv')

# LỌC CỰC KỲ QUAN TRỌNG: Chỉ lấy những run mà eeg_content_present = True (đã tải ruột)
cand = run_summary[run_summary['eeg_content_present'].astype(bool)].copy()

if len(cand) == 0:
    print("❌ CẢNH BÁO: Bảng summary báo không có run nào chứa nội dung .eeg (toàn xác rỗng). Bạn cần tải .eeg về máy trước!")
else:
    print(f"✅ Bảng summary xác nhận có {len(cand)} file ĐÃ ĐƯỢC TẢI DỮ LIỆU. Bắt đầu xử lý...\n")

# =========================================================================
# 2. XỬ LÝ HÀNG LOẠT TRÊN CÁC FILE HỢP LỆ
# =========================================================================
WINDOW_SEC = 2.0  
STEP_SEC = 1.0    

all_features = [] # List tổng

# Duyệt qua các run hợp lệ (đã có ruột)
for idx, row in cand.iterrows():
    base = str(row['base'])
    vhdr_path = Path(base + '.vhdr')
    run_name = vhdr_path.stem
    
    if not vhdr_path.exists():
        print(f"⚠️ Bỏ qua {run_name} (Có trong summary nhưng file thực tế bị mất)")
        continue
        
    print(f"Đang xử lý: {run_name} ...")
    
    # 2.1 Load và Denoise (giống hệt Notebook 04 gốc)
    raw = mne.io.read_raw_brainvision(vhdr_path, preload=True, verbose='ERROR')
    sfreq = raw.info['sfreq']
    
    raw.pick_types(eeg=True, ecog=True, meg=False)
    # Lọc Notch và Bandpass
    raw.notch_filter(freqs=np.arange(60, 150, 60), verbose=False)
    raw.filter(l_freq=0.5, h_freq=150.0, verbose=False)
    
    data = raw.get_data()
    n_samples = data.shape[1]
    
    win_samples = int(WINDOW_SEC * sfreq)
    step_samples = int(STEP_SEC * sfreq)
    
    # 2.2 Sliding Window & Feature Extraction
    file_rows = []
    for start_idx in range(0, n_samples - win_samples + 1, step_samples):
        end_idx = start_idx + win_samples
        win = data[:, start_idx:end_idx]
        
        t_mid = ((start_idx + end_idx) / 2.0) / sfreq
        
        rms_mean = float(np.nanmean(np.sqrt(np.nanmean(win ** 2, axis=1))))
        var_mean = float(np.nanmean(np.nanvar(win, axis=1)))
        ptp_mean = float(np.nanmean(np.ptp(win, axis=1)))
        ll_mean = float(np.nanmean(np.sum(np.abs(np.diff(win, axis=1)), axis=1)))
        
        file_rows.append({
            'run_id': run_name, 
            't_mid_s': t_mid, 
            'rms': rms_mean,
            'variance': var_mean,
            'ptp': ptp_mean,
            'line_length': ll_mean
        })
    
    all_features.extend(file_rows)
    print(f"  -> Trích xuất xong {len(file_rows)} cửa sổ.")

print("-" * 50)

# =========================================================================
# 3. TẠO LAG FEATURES AN TOÀN THEO TỪNG FILE
# =========================================================================
if len(all_features) > 0:
    df_features = pd.DataFrame(all_features)
    feature_cols = ['rms', 'variance', 'ptp', 'line_length']

    for col in feature_cols:
        df_features[f'{col}_lag1'] = df_features.groupby('run_id')[col].shift(1)
        df_features[f'{col}_lag2'] = df_features.groupby('run_id')[col].shift(2)

    df_features = df_features.dropna().reset_index(drop=True)

    print(f"✅ Hoàn tất! Tổng dữ liệu: {len(df_features)} dòng.")

    # Lưu file đầu ra
    output_file = paths.outputs_dir / "ds003029_forecasting_features_multirun.csv"
    df_features.to_csv(output_file, index=False)
    display(df_features.head())

✅ Bảng summary xác nhận có 15 file ĐÃ ĐƯỢC TẢI DỮ LIỆU. Bắt đầu xử lý...

Đang xử lý: sub-jh101_ses-presurgery_task-ictal_acq-ecog_run-04_ieeg ...
NOTE: pick_types() is a legacy function. New code should use inst.pick(...).
  -> Trích xuất xong 152 cửa sổ.
Đang xử lý: sub-pt01_ses-presurgery_task-ictal_acq-ecog_run-03_ieeg ...
NOTE: pick_types() is a legacy function. New code should use inst.pick(...).
  -> Trích xuất xong 370 cửa sổ.
Đang xử lý: sub-pt01_ses-presurgery_task-ictal_acq-ecog_run-04_ieeg ...
NOTE: pick_types() is a legacy function. New code should use inst.pick(...).
  -> Trích xuất xong 310 cửa sổ.
Đang xử lý: sub-pt11_ses-presurgery_task-ictal_acq-ecog_run-01_ieeg ...
NOTE: pick_types() is a legacy function. New code should use inst.pick(...).
  -> Trích xuất xong 180 cửa sổ.
Đang xử lý: sub-pt11_ses-presurgery_task-ictal_acq-ecog_run-02_ieeg ...
NOTE: pick_types() is a legacy function. New code should use inst.pick(...).
  -> Trích xuất xong 248 cửa sổ.
Đang xử lý: sub

,run_id,t_mid_s,rms,variance,ptp,line_length,rms_lag1,rms_lag2,variance_lag1,variance_lag2,ptp_lag1,ptp_lag2,line_length_lag1,line_length_lag2
0,sub-jh101_ses-presurgery_task-ictal_acq-ecog_r...,3.0,0.000856,0.000069,0.002836,0.162605,0.001270,0.001476,0.000090,0.000098,0.005034,0.006154,0.112155,0.115758
1,sub-jh101_ses-presurgery_task-ictal_acq-ecog_r...,4.0,0.000830,0.000067,0.002821,0.167604,0.000856,0.001270,0.000069,0.000090,0.002836,0.005034,0.162605,0.112155
2,sub-jh101_ses-presurgery_task-ictal_acq-ecog_r...,5.0,0.000959,0.000069,0.004918,0.170931,0.000830,0.000856,0.000067,0.000069,0.002821,0.002836,0.167604,0.162605
3,sub-jh101_ses-presurgery_task-ictal_acq-ecog_r...,6.0,0.001364,0.000107,0.004917,0.181933,0.000959,0.000830,0.000069,0.000067,0.004918,0.002821,0.170931,0.167604
4,sub-jh101_ses-presurgery_task-ictal_acq-ecog_r...,7.0,0.001366,0.000104,0.005028,0.178711,0.001364,0.000959,0.000107,0.000069,0.004917,0.004918,0.181933,0.170931


In [6]:
# =========================================================================
# 2. TIỀN XỬ LÝ VÀ TRÍCH XUẤT ĐẶC TRƯNG HÀNG LOẠT
# =========================================================================
WINDOW_SEC = 2.0  
STEP_SEC = 1.0    

all_features = [] # Danh sách tổng chứa dữ liệu của mọi bệnh nhân

# Duyệt qua từng bản ghi hợp lệ
for idx, row in cand.iterrows():
    base = str(row['base'])
    vhdr_path = Path(base + '.vhdr')
    run_name = vhdr_path.stem
    
    if not vhdr_path.exists():
        print(f"⚠️ Bỏ qua {run_name} (Có trong summary nhưng không tìm thấy file vhdr thực tế)")
        continue
        
    print(f"Đang xử lý: {run_name} ...")
    
    # Load và Denoise
    # MNE thỉnh thoảng hiện cảnh báo hàm cũ (pick_types), ta truyền verbose='ERROR' để ẩn bớt warning rác
    raw = mne.io.read_raw_brainvision(vhdr_path, preload=True, verbose='ERROR')
    sfreq = raw.info['sfreq']
    
    raw.pick_types(eeg=True, ecog=True, meg=False)
    
    # Lọc Notch và Bandpass
    raw.notch_filter(freqs=np.arange(60, 150, 60), verbose=False)
    raw.filter(l_freq=0.5, h_freq=150.0, verbose=False)
    
    data = raw.get_data()
    n_samples = data.shape[1]
    
    # Đổi giây ra số lượng điểm dữ liệu
    win_samples = int(WINDOW_SEC * sfreq)
    step_samples = int(STEP_SEC * sfreq)
    
    # Sliding Window & Feature Extraction
    file_rows = []
    for start_idx in range(0, n_samples - win_samples + 1, step_samples):
        end_idx = start_idx + win_samples
        win = data[:, start_idx:end_idx]
        
        t_mid = ((start_idx + end_idx) / 2.0) / sfreq
        
        # Tính toán các đặc trưng (lấy trung bình qua toàn bộ các kênh)
        rms_mean = float(np.nanmean(np.sqrt(np.nanmean(win ** 2, axis=1))))
        var_mean = float(np.nanmean(np.nanvar(win, axis=1)))
        ptp_mean = float(np.nanmean(np.ptp(win, axis=1)))
        ll_mean = float(np.nanmean(np.sum(np.abs(np.diff(win, axis=1)), axis=1)))
        
        file_rows.append({
            'run_id': run_name, 
            't_mid_s': t_mid, 
            'rms': rms_mean,
            'variance': var_mean,
            'ptp': ptp_mean,
            'line_length': ll_mean
        })
    
    all_features.extend(file_rows)
    print(f"  -> Trích xuất xong {len(file_rows)} cửa sổ.")

print("-" * 50)
print(f"Tổng số cửa sổ trích xuất được từ toàn bộ dataset: {len(all_features)}")

Đang xử lý: sub-jh101_ses-presurgery_task-ictal_acq-ecog_run-04_ieeg ...
NOTE: pick_types() is a legacy function. New code should use inst.pick(...).
  -> Trích xuất xong 152 cửa sổ.
Đang xử lý: sub-pt01_ses-presurgery_task-ictal_acq-ecog_run-03_ieeg ...
NOTE: pick_types() is a legacy function. New code should use inst.pick(...).
  -> Trích xuất xong 370 cửa sổ.
Đang xử lý: sub-pt01_ses-presurgery_task-ictal_acq-ecog_run-04_ieeg ...
NOTE: pick_types() is a legacy function. New code should use inst.pick(...).
  -> Trích xuất xong 310 cửa sổ.
Đang xử lý: sub-pt11_ses-presurgery_task-ictal_acq-ecog_run-01_ieeg ...
NOTE: pick_types() is a legacy function. New code should use inst.pick(...).
  -> Trích xuất xong 180 cửa sổ.
Đang xử lý: sub-pt11_ses-presurgery_task-ictal_acq-ecog_run-02_ieeg ...
NOTE: pick_types() is a legacy function. New code should use inst.pick(...).
  -> Trích xuất xong 248 cửa sổ.
Đang xử lý: sub-pt13_ses-presurgery_task-ictal_acq-ecog_run-03_ieeg ...
NOTE: pick_types(

### Khởi tạo Lag Features cho bài toán Dự báo
Sử dụng hàm `.shift()` của Pandas để đẩy lùi dữ liệu, tạo ra các cột giá trị ở các bước thời gian trước đó. Những hàng đầu tiên bị thiếu dữ liệu (NaN) do việc dịch chuyển sẽ được loại bỏ để đảm bảo chất lượng cho mô hình ARIMA/Hồi quy sau này.

In [7]:
# =========================================================================
# 3. TẠO LAG FEATURES AN TOÀN THEO TỪNG FILE VÀ LƯU KẾT QUẢ
# =========================================================================
if len(all_features) > 0:
    df_features = pd.DataFrame(all_features)
    feature_cols = ['rms', 'variance', 'ptp', 'line_length']

    # Tạo Lags (t-1, t-2) nội bộ trong từng run_id
    for col in feature_cols:
        df_features[f'{col}_lag1'] = df_features.groupby('run_id')[col].shift(1)
        df_features[f'{col}_lag2'] = df_features.groupby('run_id')[col].shift(2)

    # Loại bỏ các hàng bị thiếu dữ liệu (NaN) ở những giây đầu tiên của mỗi file
    df_features = df_features.dropna().reset_index(drop=True)

    print(f"✅ Hoàn tất quá trình tạo Lags! Dữ liệu sẵn sàng cho Forecasting: {len(df_features)} dòng.")

    # Lưu file đầu ra
    output_file = paths.outputs_dir / "ds003029_forecasting_features_multirun.csv"
    df_features.to_csv(output_file, index=False)
    
    # Hiển thị vài dòng mẫu
    display(df_features.head())

✅ Hoàn tất quá trình tạo Lags! Dữ liệu sẵn sàng cho Forecasting: 5237 dòng.


,run_id,t_mid_s,rms,variance,ptp,line_length,rms_lag1,rms_lag2,variance_lag1,variance_lag2,ptp_lag1,ptp_lag2,line_length_lag1,line_length_lag2
0,sub-jh101_ses-presurgery_task-ictal_acq-ecog_r...,3.0,0.000856,0.000069,0.002836,0.162605,0.001270,0.001476,0.000090,0.000098,0.005034,0.006154,0.112155,0.115758
1,sub-jh101_ses-presurgery_task-ictal_acq-ecog_r...,4.0,0.000830,0.000067,0.002821,0.167604,0.000856,0.001270,0.000069,0.000090,0.002836,0.005034,0.162605,0.112155
2,sub-jh101_ses-presurgery_task-ictal_acq-ecog_r...,5.0,0.000959,0.000069,0.004918,0.170931,0.000830,0.000856,0.000067,0.000069,0.002821,0.002836,0.167604,0.162605
3,sub-jh101_ses-presurgery_task-ictal_acq-ecog_r...,6.0,0.001364,0.000107,0.004917,0.181933,0.000959,0.000830,0.000069,0.000067,0.004918,0.002821,0.170931,0.167604
4,sub-jh101_ses-presurgery_task-ictal_acq-ecog_r...,7.0,0.001366,0.000104,0.005028,0.178711,0.001364,0.000959,0.000107,0.000069,0.004917,0.004918,0.181933,0.170931
